In [1]:
os.chdir(r"C:\Users\Admin\Desktop\hospoda.CZECH-PUB\code\czech-pub")

In [2]:
import pandas as pd
import json
from pathlib import Path
pd.set_option("display.max_colwidth", None)

## loading the data

In [3]:
# loading the data
df = pd.read_csv(r"data\implicature\circa_google\circa-data.tsv", sep="\t")

In [20]:
# we wrap the given contexts into short text labels to refer to them and sample the dataset according to them
context_map = {
    "X wants to know about Y's food preferences.": {
        "label": "food",
        "cz": "X se chce dozvědět, co Y nejraději jí."
    },
    "X wants to know what activities Y likes to do during weekends.": {
        "label": "weekend",
        "cz": "X chce vědět, co Y rád dělá o víkendech."
    },
    "X wants to know what sorts of books Y likes to read.": {
        "label": "books",
        "cz": "X chce vědět, jaké knížky Y rád čte."
    },
    "Y has just moved into a neighbourhood and meets his/her new neighbour X.": {
        "label": "new_neighbor",
        "cz": "Y se právě přistěhoval do sousedství a potkává svého nového souseda X."
    },
    "X and Y are colleagues who are leaving work on a Friday at the same time.": {
        "label": "friday_coworkers",
        "cz": "X a Y jsou kolegové, kteří v pátek odcházejí z práce ve stejnou chvíli."
    },
    "X wants to know about Y's music preferences.": {
        "label": "music",
        "cz": "X chce vědět, jakou hudbu Y rád poslouchá."
    },
    "Y has just travelled from a different city to meet X.": {
        "label": "travel_visit",
        "cz": "Y právě přijel z jiného města, aby se viděl s X."
    },
    "X and Y are childhood neighbours who unexpectedly run into each other at a cafe.": {
        "label": "childhood_neighbors",
        "cz": "X a Y jsou sousedé z dětství, kteří na sebe náhodou narazili v kavárně."
    },
    "Y has just told X that he/she is thinking of buying a flat in New York.": {
        "label": "ny_flat",
        "cz": "Y právě řekl X, že uvažuje o koupi bytu v New Yorku."
    },
    "Y has just told X that he/she is considering switching his/her job.": {
        "label": "job_switch",
        "cz": "Y právě řekl X, že zvažuje změnu práce."
    }
} # X wants to know about Y's music preferences." "X chce vědět, jakou hudbu Y rád poslouchá."
#  "Y has just moved into a neighbourhood and meets his/her new neighbour X." "Y se právě přistěhoval do sousedství a potkává svého nového souseda X."

# Y has just told X that he/she is thinking of buying a flat in New York.
# Y právě řekl X, že uvažuje o koupi bytu v New Yorku.



#"Y has just travelled from a different city to meet X."
#"Y právě přijel z jiného města, aby se viděl s X."

# "X wants to know what activities Y likes to do during weekends."
# X chce vědět, co Y rád dělá o víkendech.


In [21]:
# see some examples of data
df.head(30)

,id,context,question-X,answer-Y,goldstandard1,context_label,context_cz
0,0,Y has just travelled from a different city to meet X.,Are you employed?,I'm a veterinary technician.,Yes,travel_visit,"Y právě přijel z jiného města, aby se viděl s X."
1,1,X wants to know about Y's food preferences.,Are you a fan of Korean food?,I wouldn't say so,No,food,"X se chce dozvědět, co Y nejraději jí."
2,2,Y has just told X that he/she is thinking of buying a flat in New York.,Are you bringing any pets into the flat?,I do not own any pets,No,ny_flat,"Y právě řekl X, že uvažuje o koupi bytu v New Yorku."
3,3,X wants to know what activities Y likes to do during weekends.,Would you like to get some fresh air in your free time?,I am desperate to get out of the city.,Yes,weekend,"X chce vědět, co Y rád podniká o víkendech."
5,5,X wants to know what sorts of books Y likes to read.,Do you like to read self-help books?,I'm not a fan of them,No,books,"X chce vědět, jaké knížky Y rád čte."
6,6,X wants to know about Y's food preferences.,Do you enjoy foreign cuisine?,I like many cuisines.,Yes,food,"X se chce dozvědět, co Y nejraději jí."
7,7,Y has just travelled from a different city to meet X.,Is your new job going well?,I love what I do.,Yes,travel_visit,"Y právě přijel z jiného města, aby se viděl s X."
8,8,X wants to know what sorts of books Y likes to read.,Are long books your thing?,I rarely read any other type of book.,Yes,books,"X chce vědět, jaké knížky Y rád čte."
9,9,X wants to know about Y's food preferences.,Have you had pizza recently,My husband ordered some last night.,Yes,food,"X se chce dozvědět, co Y nejraději jí."
12,12,X wants to know what activities Y likes to do during weekends.,Do you enjoy playing any sports?,Basketball is a lot of fun.,Yes,weekend,"X chce vědět, co Y rád podniká o víkendech."


In [6]:
# size of the dataset
df.shape

(34268, 8)

In [7]:
# printing info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34268 entries, 0 to 34267
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             34268 non-null  int64 
 1   context        34268 non-null  object
 2   question-X     34268 non-null  object
 3   canquestion-X  34258 non-null  object
 4   answer-Y       34268 non-null  object
 5   judgements     34268 non-null  object
 6   goldstandard1  31525 non-null  object
 7   goldstandard2  33497 non-null  object
dtypes: int64(1), object(7)
memory usage: 2.1+ MB


## cleaning + analysis


In [8]:
# checking missing values
print(df.isna().sum())

id                  0
context             0
question-X          0
canquestion-X      10
answer-Y            0
judgements          0
goldstandard1    2743
goldstandard2     771
dtype: int64


In [9]:
# leaving only needed columns
df = df[['id', 'context', 'question-X', 'answer-Y', 'goldstandard1']]

In [10]:
# keeping only rows where the correct answer is non NaN
df = df.dropna(subset=['goldstandard1'])
print("shape: ", df.shape)
df.head()

shape:  (31525, 5)


,id,context,question-X,answer-Y,goldstandard1
0,0,Y has just travelled from a different city to meet X.,Are you employed?,I'm a veterinary technician.,Yes
1,1,X wants to know about Y's food preferences.,Are you a fan of Korean food?,I wouldn't say so,No
2,2,Y has just told X that he/she is thinking of buying a flat in New York.,Are you bringing any pets into the flat?,I do not own any pets,No
3,3,X wants to know what activities Y likes to do during weekends.,Would you like to get some fresh air in your free time?,I am desperate to get out of the city.,Yes
4,4,X and Y are childhood neighbours who unexpectedly run into each other at a cafe.,Is your family still living in the neighborhood?,My parents are snowbirds now.,"In the middle, neither yes nor no"


In [11]:
# preserving only cases where the direct answer is either Yes or No
df = df[df["goldstandard1"].isin(["Yes", "No"])].copy()
print("shape: ", df.shape)
df.head()

shape:  (25333, 5)


,id,context,question-X,answer-Y,goldstandard1
0,0,Y has just travelled from a different city to meet X.,Are you employed?,I'm a veterinary technician.,Yes
1,1,X wants to know about Y's food preferences.,Are you a fan of Korean food?,I wouldn't say so,No
2,2,Y has just told X that he/she is thinking of buying a flat in New York.,Are you bringing any pets into the flat?,I do not own any pets,No
3,3,X wants to know what activities Y likes to do during weekends.,Would you like to get some fresh air in your free time?,I am desperate to get out of the city.,Yes
5,5,X wants to know what sorts of books Y likes to read.,Do you like to read self-help books?,I'm not a fan of them,No


## analyzing context distribution


In [12]:
# showing distribution of the unique contexts
print("\nnumber of unique contexts:", df["context"].nunique())
print("\ncontext counts:")
print(df["context"].value_counts())


number of unique contexts: 10

context counts:
context
X and Y are childhood neighbours who unexpectedly run into each other at a cafe.    2804
Y has just travelled from a different city to meet X.                               2656
X and Y are colleagues who are leaving work on a Friday at the same time.           2652
Y has just moved into a neighbourhood and meets his/her new neighbour X.            2601
Y has just told X that he/she is considering switching his/her job.                 2515
Y has just told X that he/she is thinking of buying a flat in New York.             2514
X wants to know about Y's music preferences.                                        2488
X wants to know what activities Y likes to do during weekends.                      2454
X wants to know what sorts of books Y likes to read.                                2376
X wants to know about Y's food preferences.                                         2273
Name: count, dtype: int64


In [13]:
# labeling the groups
df["context_label"] = df["context"].map(lambda x: context_map[x]["label"])
df["context_cz"] = df["context"].map(lambda x: context_map[x]["cz"])
df.head()

,id,context,question-X,answer-Y,goldstandard1,context_label,context_cz
0,0,Y has just travelled from a different city to meet X.,Are you employed?,I'm a veterinary technician.,Yes,travel_visit,"Y právě přijel z jiného města, aby se viděl s X."
1,1,X wants to know about Y's food preferences.,Are you a fan of Korean food?,I wouldn't say so,No,food,"X se chce dozvědět, co Y nejraději jí."
2,2,Y has just told X that he/she is thinking of buying a flat in New York.,Are you bringing any pets into the flat?,I do not own any pets,No,ny_flat,"Y právě řekl X, že uvažuje o koupi bytu v New Yorku."
3,3,X wants to know what activities Y likes to do during weekends.,Would you like to get some fresh air in your free time?,I am desperate to get out of the city.,Yes,weekend,"X chce vědět, co Y rád podniká o víkendech."
5,5,X wants to know what sorts of books Y likes to read.,Do you like to read self-help books?,I'm not a fan of them,No,books,"X chce vědět, jaké knížky Y rád čte."


In [14]:
print("\nunmapped contexts:")
print(df.loc[df["context_label"].isna(), "context"].drop_duplicates().tolist())


unmapped contexts:
[]


## sampling the items

In [15]:
# sorting the dataset to make the selection reproducible
df_sorted = df.sort_values(["context_label", "id"]).copy()
df_sorted.info()

<class 'pandas.core.frame.DataFrame'>
Index: 25333 entries, 5 to 34263
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             25333 non-null  int64 
 1   context        25333 non-null  object
 2   question-X     25333 non-null  object
 3   answer-Y       25333 non-null  object
 4   goldstandard1  25333 non-null  object
 5   context_label  25333 non-null  object
 6   context_cz     25333 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.5+ MB


In [16]:
# taking the first 55 items from each context group

n_per_group = 55

sample_df = (
    df_sorted.groupby("context_label", group_keys=False)
    .head(n_per_group)
    .reset_index(drop=True)
)

sample_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 550 entries, 0 to 549
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             550 non-null    int64 
 1   context        550 non-null    object
 2   question-X     550 non-null    object
 3   answer-Y       550 non-null    object
 4   goldstandard1  550 non-null    object
 5   context_label  550 non-null    object
 6   context_cz     550 non-null    object
dtypes: int64(1), object(6)
memory usage: 30.2+ KB


In [17]:
# selecting the fields for translation dataset
export_df = sample_df[
    ["id", "context_label", "question-X", "answer-Y"]
].copy()

# renaming columns for cleaner json
export_df = export_df.rename(columns={
    "question-X": "question_X",
    "answer-Y": "answer_Y"
})

# checking the result
print(export_df.head())
print(export_df.shape)

   id context_label                                  question_X  \
0   5         books        Do you like to read self-help books?   
1   8         books                  Are long books your thing?   
2  23         books      Did you buy any sci-fi books recently?   
3  55         books                  Do you enjoy sci-fi books?   
4  76         books  Was that biography that you read any good?   

                                answer_Y  
0                  I'm not a fan of them  
1  I rarely read any other type of book.  
2          I picked up two last weekend.  
3    My husband has gotten me into them.  
4          I thought it was forgettable.  
(550, 4)


In [19]:
# we will generate 10 chunkns (1 from each context group)

# defining the chunk folder
out_dir = Path(r"data\implicature\circa_google\chunked_550_eng")
out_dir.mkdir(parents=True, exist_ok=True)

# saving one chunk per context group
for context_label, group in export_df.groupby("context_label", sort=True):

    # taking each chunk
    chunk = group.drop(columns=["context_label"]).to_dict(orient="records")

    # saving the chunk
    with open(out_dir / f"circa_{context_label}.json", "w", encoding="utf-8") as f:
        json.dump(chunk, f, ensure_ascii=False, indent=2)

    print(f"created {context_label}: {len(chunk)} rows")

created books: 55 rows
created childhood_neighbors: 55 rows
created food: 55 rows
created friday_coworkers: 55 rows
created job_switch: 55 rows
created music: 55 rows
created new_neighbor: 55 rows
created ny_flat: 55 rows
created travel_visit: 55 rows
created weekend: 55 rows


In [ ]:
# here i generate empty jsons in the corresponding cz directory to paste the manual translations there.
# defining the source and target folders
src_dir = Path(r"data\implicature\circa_google\chunked_550_eng")
dst_dir = Path(r"data\implicature\circa_google\chunked_550_cz")

# creating the target folder
dst_dir.mkdir(parents=True, exist_ok=True)

# creating empty files with the same names
for path in src_dir.glob("*.json"):
    (dst_dir / path.name).write_text("", encoding="utf-8")

print("done")

In [ ]:
# ONCE TRANSLATED CHUNKS ARE READY, WE PROCEED BELOW ...

## mapping the translated dataset

In [22]:
# loading all translated chunk files
folder = Path(r"data\implicature\circa_google\chunked_550_cz")
translated_records = []

for path in sorted(folder.glob("*.json")):
    with open(path, "r", encoding="utf-8-sig") as f:
        chunk_records = json.load(f)
        translated_records.extend(chunk_records)

# building the translated dataframe
translated_df = pd.DataFrame(translated_records)

# checking what was loaded
print("translated rows:", len(translated_df))
print("unique translated ids:", translated_df["id"].nunique())
print("columns:", translated_df.columns.tolist())
translated_df.head()

translated rows: 550
unique translated ids: 550
['id', 'question_X', 'answer_Y', 'question_X_cz', 'answer_Y_cz']


,id,question_X,answer_Y,question_X_cz,answer_Y_cz
0,5,Do you like to read self-help books?,I'm not a fan of them,Čteš rád motivační knížky?,Zrovna jejich fanoušek nejsem.
1,8,Are long books your thing?,I rarely read any other type of book.,Baví tě tlusté knihy?,Skoro žádný jiný typ knížek ani nečtu.
2,23,Did you buy any sci-fi books recently?,I picked up two last weekend.,Koupil jsi si v poslední době nějaké sci-fi?,Minulý víkend jsem si dvě pořídil.
3,55,Do you enjoy sci-fi books?,My husband has gotten me into them.,Baví tě sci-fi knížky?,Můj manžel mě k nim přivedl.
4,76,Was that biography that you read any good?,I thought it was forgettable.,"Byla ta biografie, co jsi četl, dobrá?",Přišla mi celkem o ničem.


In [24]:
# merging the translations back to the sampled dataset
final_df = sample_df.merge(
    translated_df[["id", "question_X_cz", "answer_Y_cz"]],
    on="id",
    how="left",
    validate="one_to_one"
)

# checking whether everything matched
print("sample rows:", len(sample_df))
print("final rows:", len(final_df))
print("missing question translations:", final_df["question_X_cz"].isna().sum())
print("missing answer translations:", final_df["answer_Y_cz"].isna().sum())
print("nans:\n", final_df.isna().sum())

final_df.head()

sample rows: 550
final rows: 550
missing question translations: 0
missing answer translations: 0
nans: id               0
context          0
question-X       0
answer-Y         0
goldstandard1    0
context_label    0
context_cz       0
question_X_cz    0
answer_Y_cz      0
dtype: int64


,id,context,question-X,answer-Y,goldstandard1,context_label,context_cz,question_X_cz,answer_Y_cz
0,5,X wants to know what sorts of books Y likes to read.,Do you like to read self-help books?,I'm not a fan of them,No,books,"X chce vědět, jaké knížky Y rád čte.",Čteš rád motivační knížky?,Zrovna jejich fanoušek nejsem.
1,8,X wants to know what sorts of books Y likes to read.,Are long books your thing?,I rarely read any other type of book.,Yes,books,"X chce vědět, jaké knížky Y rád čte.",Baví tě tlusté knihy?,Skoro žádný jiný typ knížek ani nečtu.
2,23,X wants to know what sorts of books Y likes to read.,Did you buy any sci-fi books recently?,I picked up two last weekend.,Yes,books,"X chce vědět, jaké knížky Y rád čte.",Koupil jsi si v poslední době nějaké sci-fi?,Minulý víkend jsem si dvě pořídil.
3,55,X wants to know what sorts of books Y likes to read.,Do you enjoy sci-fi books?,My husband has gotten me into them.,Yes,books,"X chce vědět, jaké knížky Y rád čte.",Baví tě sci-fi knížky?,Můj manžel mě k nim přivedl.
4,76,X wants to know what sorts of books Y likes to read.,Was that biography that you read any good?,I thought it was forgettable.,No,books,"X chce vědět, jaké knížky Y rád čte.","Byla ta biografie, co jsi četl, dobrá?",Přišla mi celkem o ničem.


## generating distractors

In [25]:
# for distractor creation, we will create chunks with fields needed for distractor creation, and then empty jsons to fill it in with distractors

# defining the base folder
base_dir = Path(r"data\implicature\circa_google\distractor_creation")

# defining the output folders
without_dir = base_dir / "without_distractors"
with_dir = base_dir / "with_distractors"

# creating the folders
without_dir.mkdir(parents=True, exist_ok=True)
with_dir.mkdir(parents=True, exist_ok=True)

# saving one json per context group without distractors
for context_label, group in final_df.groupby("context_label", sort=True):
    # selecting the fields for distractor creation
    chunk_df = group[["id", "context_cz", "question_X_cz", "answer_Y_cz"]].copy()

    # converting the chunk to list of dicts
    chunk = chunk_df.to_dict(orient="records")

    # defining the file names
    without_name = f"circa_{context_label}.json"
    with_name = f"circa_{context_label}_with_distractors.json"

    # saving the source chunk
    with open(without_dir / without_name, "w", encoding="utf-8") as f:
        json.dump(chunk, f, ensure_ascii=False, indent=2)

    # creating the empty target file
    (with_dir / with_name).write_text("", encoding="utf-8")

    print(f"saved: {without_name} ({len(chunk)} rows)")
    print(f"created empty: {with_name}")

saved: circa_books.json (55 rows)
created empty: circa_books_with_distractors.json
saved: circa_childhood_neighbors.json (55 rows)
created empty: circa_childhood_neighbors_with_distractors.json
saved: circa_food.json (55 rows)
created empty: circa_food_with_distractors.json
saved: circa_friday_coworkers.json (55 rows)
created empty: circa_friday_coworkers_with_distractors.json
saved: circa_job_switch.json (55 rows)
created empty: circa_job_switch_with_distractors.json
saved: circa_music.json (55 rows)
created empty: circa_music_with_distractors.json
saved: circa_new_neighbor.json (55 rows)
created empty: circa_new_neighbor_with_distractors.json
saved: circa_ny_flat.json (55 rows)
created empty: circa_ny_flat_with_distractors.json
saved: circa_travel_visit.json (55 rows)
created empty: circa_travel_visit_with_distractors.json
saved: circa_weekend.json (55 rows)
created empty: circa_weekend_with_distractors.json


In [ ]:
# distractors are generated outside of the notebook, they are manually reviewed and loaded to jsons

In [36]:
# defining the folder with completed distractor files
with_dir = Path(r"data\implicature\circa_google\distractor_creation\with_distractors")

# loading all distractor records
all_records = []

for path in sorted(with_dir.glob("*.json")):
    with open(path, "r", encoding="utf-8-sig") as f:
        chunk_records = json.load(f)
        all_records.extend(chunk_records)

# building the distractor dataframe
distractor_df = pd.DataFrame(all_records)

# checking the result
print("rows loaded:", len(distractor_df))
print("unique ids:", distractor_df["id"].nunique())
print("duplicate ids:", distractor_df["id"].duplicated().sum())
print(distractor_df.columns.tolist())
distractor_df.head()

rows loaded: 550
unique ids: 550
duplicate ids: 0
['id', 'context_cz', 'question_X_cz', 'answer_Y_cz', 'direct_answer_pos', 'direct_answer_neg', 'distractor_lexical', 'distractor_associative']


,id,context_cz,question_X_cz,answer_Y_cz,direct_answer_pos,direct_answer_neg,distractor_lexical,distractor_associative
0,5,"X chce vědět, jaké knížky Y rád čte.",Čteš rád motivační knížky?,Zrovna jejich fanoušek nejsem.,"Ano, čtu rád motivační knížky.","Ne, nečtu rád motivační knížky.",Zrovna jejich názvy si hledám v online katalogu.,Motivační literatura se často zaměřuje na kariérní růst.
1,8,"X chce vědět, jaké knížky Y rád čte.",Baví tě tlusté knihy?,Skoro žádný jiný typ knížek ani nečtu.,"Ano, baví mě tlusté knihy.","Ne, nebaví mě tlusté knihy.",Skoro žádný jiný regál v knihovně není tak plný.,Počet stran v knize často ovlivňuje její konečnou váhu.
2,23,"X chce vědět, jaké knížky Y rád čte.",Koupil jsi si v poslední době nějaké sci-fi?,Minulý víkend jsem si dvě pořídil.,"Ano, koupil jsem si v poslední době nějaké sci-fi.","Ne, nekoupil jsem si v poslední době žádné sci-fi.",Minulý víkend jsem v jedné staré knize dlouho listoval.,Vědeckofantastické romány se často odehrávají v daleké budoucnosti.
3,55,"X chce vědět, jaké knížky Y rád čte.",Baví tě sci-fi knížky?,Můj manžel mě k nim přivedl.,"Ano, baví mě sci-fi knížky.","Ne, nebaví mě sci-fi knížky.",Můj manžel mě do té nové knihovny včera doprovodil.,Technologický pokrok je častým tématem moderní sci-fi prózy.
4,76,"X chce vědět, jaké knížky Y rád čte.","Byla ta biografie, co jsi četl, dobrá?",Přišla mi celkem o ničem.,"Ano, byla dobrá.","Ne, nebyla dobrá.",Přišla mi včera domů zásilka s objednanými romány.,Životopisy slavných lidí se často drží na seznamech bestsellerů.


In [33]:
distractor_df[distractor_df["associative_distractor"].notna() ]

,id,context_cz,question_X_cz,answer_Y_cz,direct_answer_pos,direct_answer_neg,distractor_lexical,distractor_associative,associative_distractor
377,747,Y se právě přistěhoval do sousedství a potkává svého nového souseda X.,Máte manželku nebo partnerku?,Se svou ženou jsem ženatý už pět let.,"Ano, mám.","Ne, nemám.",Se svou novou sekačkou jsem včera pracoval tři hodiny.,NaN,Zlatnictví nabízejí široký výběr snubních prstenů.


In [38]:
# keeping only the needed generated fields
distractor_df = distractor_df[
    [
        "id",
        "direct_answer_pos",
        "direct_answer_neg",
        "distractor_lexical",
        "distractor_associative",
    ]
].copy()

In [39]:
# merging the distractors back to the translated dataset
assembled_df = final_df.merge(
    distractor_df,
    on="id",
    how="left",
    validate="one_to_one"
)

# checking the merge result
print("rows in final_df:", len(final_df))
print("rows in assembled_df:", len(assembled_df))
print("missing direct_answer_pos:", assembled_df["direct_answer_pos"].isna().sum())
print("missing direct_answer_neg:", assembled_df["direct_answer_neg"].isna().sum())
print("missing distractor_lexical:", assembled_df["distractor_lexical"].isna().sum())
print("missing distractor_associative:", assembled_df["distractor_associative"].isna().sum())

assembled_df.head()

rows in final_df: 550
rows in assembled_df: 550
missing direct_answer_pos: 0
missing direct_answer_neg: 0
missing distractor_lexical: 0
missing distractor_associative: 0


,id,context,question-X,answer-Y,goldstandard1,context_label,context_cz,question_X_cz,answer_Y_cz,direct_answer_pos,direct_answer_neg,distractor_lexical,distractor_associative
0,5,X wants to know what sorts of books Y likes to read.,Do you like to read self-help books?,I'm not a fan of them,No,books,"X chce vědět, jaké knížky Y rád čte.",Čteš rád motivační knížky?,Zrovna jejich fanoušek nejsem.,"Ano, čtu rád motivační knížky.","Ne, nečtu rád motivační knížky.",Zrovna jejich názvy si hledám v online katalogu.,Motivační literatura se často zaměřuje na kariérní růst.
1,8,X wants to know what sorts of books Y likes to read.,Are long books your thing?,I rarely read any other type of book.,Yes,books,"X chce vědět, jaké knížky Y rád čte.",Baví tě tlusté knihy?,Skoro žádný jiný typ knížek ani nečtu.,"Ano, baví mě tlusté knihy.","Ne, nebaví mě tlusté knihy.",Skoro žádný jiný regál v knihovně není tak plný.,Počet stran v knize často ovlivňuje její konečnou váhu.
2,23,X wants to know what sorts of books Y likes to read.,Did you buy any sci-fi books recently?,I picked up two last weekend.,Yes,books,"X chce vědět, jaké knížky Y rád čte.",Koupil jsi si v poslední době nějaké sci-fi?,Minulý víkend jsem si dvě pořídil.,"Ano, koupil jsem si v poslední době nějaké sci-fi.","Ne, nekoupil jsem si v poslední době žádné sci-fi.",Minulý víkend jsem v jedné staré knize dlouho listoval.,Vědeckofantastické romány se často odehrávají v daleké budoucnosti.
3,55,X wants to know what sorts of books Y likes to read.,Do you enjoy sci-fi books?,My husband has gotten me into them.,Yes,books,"X chce vědět, jaké knížky Y rád čte.",Baví tě sci-fi knížky?,Můj manžel mě k nim přivedl.,"Ano, baví mě sci-fi knížky.","Ne, nebaví mě sci-fi knížky.",Můj manžel mě do té nové knihovny včera doprovodil.,Technologický pokrok je častým tématem moderní sci-fi prózy.
4,76,X wants to know what sorts of books Y likes to read.,Was that biography that you read any good?,I thought it was forgettable.,No,books,"X chce vědět, jaké knížky Y rád čte.","Byla ta biografie, co jsi četl, dobrá?",Přišla mi celkem o ničem.,"Ano, byla dobrá.","Ne, nebyla dobrá.",Přišla mi včera domů zásilka s objednanými romány.,Životopisy slavných lidí se často drží na seznamech bestsellerů.


## converting to the unified schema (#revisit)

In [42]:
# converting one row into the unified schema
def row_to_unified_item(row: pd.Series) -> dict:
    # building the fixed option list
    options = [
        {
            "label": "A",
            "type": "direct_answer_positive",
            "text": row["direct_answer_pos"],
        },
        {
            "label": "B",
            "type": "direct_answer_negative",
            "text": row["direct_answer_neg"],
        },
        {
            "label": "C",
            "type": "distractor_lexical",
            "text": row["distractor_lexical"],
        },
        {
            "label": "D",
            "type": "distractor_associative",
            "text": row["distractor_associative"],
        },
    ]

    # deriving the correct option type from the gold label
    if row["goldstandard1"] == "Yes":
        correct_type = "direct_answer_positive"
    elif row["goldstandard1"] == "No":
        correct_type = "direct_answer_negative"
    else:
        raise ValueError(f"unexpected goldstandard1 value: {row['goldstandard1']} for id={row['id']}")

    # finding the displayed label of the correct option
    correct_option = next(
        opt["label"] for opt in options if opt["type"] == correct_type
    )

    # building the final unified item
    item = {
        "id": f"circa-indirect_answer-{int(row['id'])}",
        "creation_method": "translation",
        "phenomenon": "implicature",
        "category": "indirect_answer_binary",
        "context": row["context_cz"],
        "utterance": f'- A: {row["question_X_cz"]}\n- B: {row["answer_Y_cz"]}',
        "question": "Co mluvčí B svou odpovědí v daném kontextu nepřímo sděluje?",
        "options": options,
        "correct_option": correct_option,
        "metadata": {
            "corpus": "CIRCA",
            "topic": row['context_label'],
        },
    }

    return item

In [48]:
from utils.prompt_builder import build_prompt
test = row_to_unified_item(assembled_df.iloc[0])
print(build_prompt(test))

Vyber, co mluvčí svou odpovědí v daném kontextu nepřímo sděluje.

Přečti si kontext, výpověď a možnosti.
Vyber jednu nejlepší odpověď.

Odpověz pouze jedním písmenem: A, B, C nebo D.

Kontext: 
 X chce vědět, jaké knížky Y rád čte.

Výpověď:
- A: Čteš rád motivační knížky?
- B: Zrovna jejich fanoušek nejsem.

Co mluvčí B svou odpovědí v daném kontextu nepřímo sděluje?

Možnosti:
A. Ano, čtu rád motivační knížky.
B. Ne, nečtu rád motivační knížky.
C. Zrovna jejich názvy si hledám v online katalogu.
D. Motivační literatura se často zaměřuje na kariérní růst.


In [49]:
# write the dataset of items to implicature/eval

# converting all rows into unified items
items = assembled_df.apply(row_to_unified_item, axis=1).tolist()

# directiries
out_dir = Path(r"data\implicature\circa_google\eval")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "eval_circa_indirect_answer_binary_550.json"

# writing the unified items to json
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

print(f"saved {len(items)} items to {out_path}")
print(items[0])

saved 550 items to data\implicature\circa_google\eval\eval_circa_indirect_answer_binary_550.json
{'id': 'circa-indirect_answer-5', 'creation_method': 'translation', 'phenomenon': 'implicature', 'category': 'indirect_answer_binary', 'context': 'X chce vědět, jaké knížky Y rád čte.', 'utterance': '- A: Čteš rád motivační knížky?\n- B: Zrovna jejich fanoušek nejsem.', 'question': 'Co mluvčí B svou odpovědí v daném kontextu nepřímo sděluje?', 'options': [{'label': 'A', 'type': 'direct_answer_positive', 'text': 'Ano, čtu rád motivační knížky.'}, {'label': 'B', 'type': 'direct_answer_negative', 'text': 'Ne, nečtu rád motivační knížky.'}, {'label': 'C', 'type': 'distractor_lexical', 'text': 'Zrovna jejich názvy si hledám v online katalogu.'}, {'label': 'D', 'type': 'distractor_associative', 'text': 'Motivační literatura se často zaměřuje na kariérní růst.'}], 'correct_option': 'B', 'metadata': {'corpus': 'CIRCA', 'topic': 'books'}}


In [ ]:
#TODO: when converting it to the general format - add "topic" in the metadata - done